In [1]:
# --- 1. LIBRERÍAS DE PROCESAMIENTO DE DATOS ---
import glob
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler # Para normalizar
import h5py as h5
# --- 2. LIBRERÍAS VISUALES ---
import matplotlib.pyplot as plt

In [3]:
#LEEMOS LOS ARCHIVOS 


# 1. Buscamos los archivos con la extensión correcta
ruta_archivos = r"C:\Users\MSI\Downloads\Proyecto_Fluidos_AI\datos_sin_procesar\*.stp" # O *.hdf5 dependiendo de cómo te los den
lista_archivos = sorted(glob.glob(ruta_archivos))

print(f"¡Se encontraron {len(lista_archivos)} archivos!")
#Primero leemos las constantes y las guardamos para luego crear los tensores finales con la suma total de dimensiones
with h5.File(lista_archivos[0], 'r') as f:

    y = f['header']['y'][()]

    probe_j = f['header']['probe_j'][()]

    dim1, dim2= f['sta']['probe_u'].shape[1], f['sta']['probe_u'].shape[2]

    dimtmp = f['sta']["utmp"].shape[1]

suma_total_dimension = 0

# 2. Bucle para leerlos
for archivo in lista_archivos:
    # Abrimos el archivo temporalmente
    with h5.File(archivo, 'r') as f:

        tamaño_actual = f['sta']['probe_u'].shape[0]

        suma_total_dimension += tamaño_actual


#Inicializar los tensores finales con la suma total de dimensiones y las dimensiones individuales
prob_u = np.zeros((suma_total_dimension, dim1, dim2), dtype=np.float32)
prob_v = np.zeros((suma_total_dimension, dim1, dim2), dtype=np.float32)
prob_w = np.zeros((suma_total_dimension, dim1, dim2), dtype=np.float32)

utmp= np.zeros((suma_total_dimension, dimtmp), dtype=np.float32)
vtmp= np.zeros((suma_total_dimension, dimtmp), dtype=np.float32)
wtmp= np.zeros((suma_total_dimension, dimtmp), dtype=np.float32)



timev = np.zeros(suma_total_dimension, dtype=np.float32)

# 3. Bucle para conocer la forma de los tensores y luego concatenarlos
print("Suma total de dimensiones:", suma_total_dimension)
print("Vector final de dimensiones:", (suma_total_dimension, dim1, dim2))
print("Vector final de dimensiones tmp:", (suma_total_dimension, dimtmp))

¡Se encontraron 106 archivos!
Suma total de dimensiones: 106000
Vector final de dimensiones: (106000, 7, 288)
Vector final de dimensiones tmp: (106000, 201)


In [4]:
#CONSTRUIMOS LOS DATOS QUE BUSCAMOS

# Nuestro "marcapáginas". Empieza en la fila 0 de nuestra caja gigante.
indice_actual = 0  
lista_archivos = sorted(lista_archivos) #Para ordenarlo alfabéticamente, aunque si ya los tienes ordenados no es necesario. Solo para asegurarnos de que el orden sea el mismo que el de la primera pasada.

# La estructura es la misma: volvemos a recorrer la misma lista en el mismo orden
for archivo in lista_archivos:
    with h5.File(archivo, 'r') as f:
        
        # 1. AHORA SÍ descargamos los datos reales de este archivo a la RAM
        # Usar [()] o [:] al final es lo que le dice a h5py que descargue los números
        datos_temporales_u = f['sta']['probe_u'][()] 
        datos_temporales_v = f['sta']['probe_v'][()] 
        datos_temporales_w = f['sta']['probe_w'][()]

        datos_temporales_timev = f['header']['timev'][()]

        datos_temporales_utmp = f['sta']['utmp'][()]
        datos_temporales_vtmp = f['sta']['vtmp'][()]
        datos_temporales_wtmp = f['sta']['wtmp'][()]

        
        # Medimos cuántas filas tiene este pedazo que acabamos de descargar
        tamaño_actual = datos_temporales_u.shape[0]
        
        # 2. METEMOS LOS DATOS EN LA CAJA GIGANTE
        # Le decimos: "En prob_u, desde la fila 'indice_actual' hasta la fila 'indice_actual + tamaño_actual', 
        # mete los datos_temporales. Y cópiame todas las demás dimensiones (:, :) igual."
        prob_u[indice_actual : indice_actual + tamaño_actual, :, :] = datos_temporales_u
        prob_v[indice_actual : indice_actual + tamaño_actual, :, :] = datos_temporales_v
        prob_w[indice_actual : indice_actual + tamaño_actual, :, :] = datos_temporales_w

        timev[indice_actual : indice_actual + tamaño_actual] = datos_temporales_timev

        utmp[indice_actual : indice_actual + tamaño_actual, :] = datos_temporales_utmp
        vtmp[indice_actual : indice_actual + tamaño_actual, :] = datos_temporales_vtmp
        wtmp[indice_actual : indice_actual + tamaño_actual, :] = datos_temporales_wtmp
        
        # 3. ACTUALIZAMOS EL MARCAPÁGINAS
        # Movemos el índice hacia adelante para que el siguiente archivo 
        # se coloque justo debajo de este y no lo aplaste.
        indice_actual += tamaño_actual

In [5]:
#NORMALIZAMOS LOS DATOS

# 1. Calculamos la media y la desviación de cada eje (y LAS GUARDAMOS)
media_u, std_u = np.mean(prob_u), np.std(prob_u)
media_v, std_v = np.mean(prob_v), np.std(prob_v)
media_w, std_w = np.mean(prob_w), np.std(prob_w)

media_utmp, std_utmp = np.mean(utmp), np.std(utmp)
media_vtmp, std_vtmp = np.mean(vtmp), np.std(vtmp)
media_wtmp, std_wtmp = np.mean(wtmp), np.std(wtmp)

# 2. Aplicamos la fórmula Z = (x - media) / std "En el sitio"
# Para U
prob_u -= media_u
prob_u /= std_u

utmp -= media_utmp
utmp /= std_utmp

# Para V
prob_v -= media_v
prob_v /= std_v

vtmp -= media_vtmp
vtmp /= std_vtmp

# Para W
prob_w -= media_w
prob_w /= std_w

wtmp -= media_wtmp
wtmp /= std_wtmp

In [6]:
import h5py as h5

# Nombre del archivo final y ruta donde se guardará
nombre_archivo_final = r"C:\Users\MSI\Downloads\Proyecto_Fluidos_AI\datos_procesados\process_data.h5"

with h5.File(nombre_archivo_final, 'w') as f_destino:
    
    # ==========================================
    # 1. CREAMOS LOS GRUPOS (CARPETAS)
    # ==========================================
    grupo_header = f_destino.create_group('header')
    grupo_sta = f_destino.create_group('sta')
    grupo_stats = f_destino.create_group('stats_zscore')

    # Abrimos el archivo original de molde
    with h5.File(lista_archivos[0], 'r') as f_origen:
        
        # ==========================================
        # 2. COPIAR EL HEADER (Excluyendo 'timev')
        # ==========================================
        for variable in f_origen['header'].keys():
            if variable == 'timev':
                continue # Saltamos el timev pequeño
            f_origen.copy(f_origen['header'][variable], grupo_header, name=variable)

    # ==========================================
    # 3. GUARDAR NUESTROS DATOS GIGANTES CONCATENADOS
    # ==========================================
    
    # Tiempo al header
    grupo_header.create_dataset('timev', data=timev)
    
    # Velocidades al sta
    grupo_sta.create_dataset('probe_u', data=prob_u)
    grupo_sta.create_dataset('probe_v', data=prob_v)
    grupo_sta.create_dataset('probe_w', data=prob_w)

    # Tmp al sta
    grupo_sta.create_dataset('utmp', data=utmp)
    grupo_sta.create_dataset('vtmp', data=vtmp)
    grupo_sta.create_dataset('wtmp', data=wtmp)

    # ==========================================
    # 5. CARPETA DE METADATOS (Z-SCORE)
    # ==========================================
    grupo_stats.create_dataset('media_u', data=media_u)
    grupo_stats.create_dataset('std_u', data=std_u)
    grupo_stats.create_dataset('media_v', data=media_v)
    grupo_stats.create_dataset('std_v', data=std_v)
    grupo_stats.create_dataset('media_w', data=media_w)
    grupo_stats.create_dataset('std_w', data=std_w)

    #Los de utmp, vtmp, wtmp
    grupo_stats.create_dataset('media_utmp', data=media_utmp)
    grupo_stats.create_dataset('std_utmp', data=std_utmp)
    grupo_stats.create_dataset('media_vtmp', data=media_vtmp)
    grupo_stats.create_dataset('std_vtmp', data=std_vtmp)
    grupo_stats.create_dataset('media_wtmp', data=media_wtmp)
    grupo_stats.create_dataset('std_wtmp', data=std_wtmp)



# Comprobación final de las "carpetas" creadas
with h5.File(nombre_archivo_final, 'r') as f:
    print("Carpetas principales de tu nuevo archivo:", list(f.keys()))

Carpetas principales de tu nuevo archivo: ['header', 'sta', 'stats_zscore']
